# Домашнее задание 3. Первые шаги в SQL и запросы к одной таблице

Задание закрепляет семинары 3 и 4: выражения SQL, свои таблицы с ограничениями и запросы
`SELECT … FROM … WHERE … ORDER BY … LIMIT …` к демобазе авиаперевозок.

**Как устроен ноутбук.** Каждая задача — описание и одна или несколько ячеек, которые начинаются
с `%%sql`. Всё, что написано в такой ячейке ниже первой строки, отправляется в PostgreSQL,
а результат показывается таблицей. Ячейка выполняется сочетанием **Shift + Enter**.

**Один запрос — одна ячейка.** Если написать в ячейке несколько запросов `SELECT`, под ней появится
результат только последнего. Там, где в задаче несколько пунктов, для каждого есть своя ячейка.

**Оценка.** Всего 10 задач на **10 баллов**, баллы указаны в заголовке каждой задачи.
Задача засчитывается, если запрос выполняется и возвращает то, что просили: нужные строки,
столбцы с указанными именами и в указанном порядке, сортировка — там, где она названа.

**Перед отправкой** выполните ноутбук целиком: меню **Run → Restart Kernel and Run All Cells…** —
под каждой ячейкой должен остаться результат. Как настроить базу и как сдать работу,
написано в файле `README.md` в корне репозитория.

### Таблицы демобазы

Таблицы демобазы лежат в схеме `bookings`, в запросах пишем имя вместе со схемой: `bookings.flights`.
Свои таблицы (задача 2) создаём в схеме `public`: `public.meals`.

> На семинарах 3 и 4 таблицы демобазы назывались `public.flights`, `public.airports`.
> Теперь вместо `public.` перед ними пишется `bookings.` — сами таблицы и данные те же.

| Таблица | Одна строка — это | Нужные столбцы |
|---|---|---|
| `bookings.airports` | аэропорт | `airport_code` — код из трёх букв, `airport_name`, `city` |
| `bookings.aircrafts` | модель самолёта | `aircraft_code`, `model`, `range` — дальность полёта, км |
| `bookings.seats` | место в салоне модели самолёта | `aircraft_code`, `seat_no`, `fare_conditions` — класс обслуживания |
| `bookings.flights` | рейс в конкретный день | `flight_id`, `flight_no`, `departure_airport`, `arrival_airport`, `status`, `scheduled_departure`, `scheduled_arrival` — по расписанию, `actual_departure`, `actual_arrival` — фактически |
| `bookings.bookings` | бронирование (в нём один или несколько билетов) | `book_ref` — номер, `book_date` — когда оформлено, `total_amount` — полная стоимость |
| `bookings.tickets` | билет пассажира | `ticket_no`, `book_ref`, `passenger_name`, `contact_data` — контакты (телефон и email) |

Полная схема демобазы со всеми столбцами, ключами и связями между таблицами — в файле
[`db/schema.md`](../db/schema.md). Связи между таблицами:

![Схема демобазы](../db/schema.svg)

Значения, которые встречаются в данных:

- **`status`** рейса — `Scheduled` (продажа открыта), `On Time` (регистрация открыта, вылет вовремя),
  `Delayed` (вылет задержан), `Departed` (в воздухе), `Arrived` (прилетел), `Cancelled` (отменён);
- **`fare_conditions`** — `Economy`, `Comfort`, `Business`;
- **`contact_data`** — контакты в формате JSON: `{"email": "…", "phone": "…"}`, у части пассажиров почты нет.

Коды аэропортов, номера бронирований, даты и пороги, по которым нужно отбирать строки, названы
в условии каждой задачи — искать их в базе не требуется.

Данные сформированы на момент 13.10.2016 17:00. У рейсов, которые к этому моменту не вылетели
или не прилетели, в столбцах `actual_departure` и `actual_arrival` стоит `NULL`.

In [ ]:
import sys; sys.path.insert(0, "../jupyter")   # магия %%sql лежит в папке jupyter репозитория
%load_ext sql_magic

### Проверка подключения
Если ячейка ниже показала пять аэропортов — всё готово к работе.

In [ ]:
%%sql
SELECT * FROM bookings.airports LIMIT 5;

# Часть 1. Выражения и свои таблицы

**Памятка**

| Что | Как пишется | Пример |
|---|---|---|
| Запрос без таблицы | `SELECT выражение AS имя;` | `SELECT 2 + 2 AS four;` |
| Момент времени и дата | `timestamp '…'`, `date '…'` | `timestamp '2016-10-13 17:00'`, `date '2016-10-13'` |
| Интервал | `interval '…'` | `interval '1 hour 30 minutes'` |
| Удалить таблицу, если она есть | `DROP TABLE IF EXISTS` | `DROP TABLE IF EXISTS public.my_trips;` |

### Задача 1. Выражения без таблицы (1 балл)

Напишите **один** запрос без таблицы, который вернёт четыре столбца:

- **carts** — сколько багажных тележек нужно, чтобы за один раз вывезти 250 чемоданов,
  если на тележку помещается 36;
- **arrival** — момент прилёта рейса, который вылетел 13 октября 2016 года в 17:00 и летит 2 часа 35 минут;
- **days_left** — сколько дней от 13 октября 2016 года до 31 декабря 2016 года;
- **both_unknown** — результат сравнения `NULL` с `NULL`, записанного так, чтобы получилось `true`, а не `NULL`.

Каждое значение должно вычисляться выражением, а не быть вписанным готовым числом.

In [ ]:
%%sql
-- Задача 1. Напишите запрос

### Задача 2. Бортовое меню и заказы питания (1,5 балла)

Авиакомпания продаёт на борту горячее питание. Нужны две таблицы в схеме `public`.

Таблица меню **meals**:
- **meal_id** — уникальный номер блюда, целое число;
- **meal_name** — название, текст, обязательный столбец;
- **price** — цена, число с 2 знаками после запятой, больше нуля.

Таблица заказов **meal_orders**:
- **order_id** — уникальный номер заказа, целое число;
- **meal_id** — номер блюда из меню, целое число, обязательный столбец;
- **order_time** — время заказа, дата и время с часовым поясом, обязательный столбец;
- **quantity** — количество порций, целое число, больше нуля.

**Что нужно сделать:**

1. Создайте обе таблицы: подберите типы данных, задайте первичные ключи, обязательность столбцов,
   проверки значений и внешний ключ из заказов на меню. Добавьте в меню три блюда:

   ```
   meal_id;meal_name;price
   1;Курица с рисом;450.00
   2;Паста с овощами;390.50
   3;Сырная тарелка;520.00
   ```

   и три заказа с существующими блюдами.
2. Выведите все строки таблицы **meals**.
3. Выведите все строки таблицы **meal_orders**.
4. Попробуйте добавить заказ на блюдо с номером 99, которого нет в меню. PostgreSQL должен ответить
   ошибкой — это и есть правильный результат. В комментарии в той же ячейке напишите одной фразой,
   какое ограничение сработало и от чего оно защищает.

In [ ]:
%%sql
-- Задача 2.1. Напишите команды создания таблиц и добавления строк

-- Эти две строки оставьте: они удаляют таблицы, если те уже созданы,
-- и ячейку можно выполнять сколько угодно раз.
DROP TABLE IF EXISTS public.meal_orders;
DROP TABLE IF EXISTS public.meals;

In [ ]:
%%sql
-- Задача 2.2. Напишите запрос

In [ ]:
%%sql
-- Задача 2.3. Напишите запрос

In [ ]:
%%sql
-- Задача 2.4. Напишите команду INSERT и комментарий: какое ограничение сработало и от чего оно защищает

# Часть 2. Запросы к одной таблице

**Памятка**

| Что | Как пишется | Пример |
|---|---|---|
| Часть даты | `date_part('часть', значение)` | `date_part('month', book_date) = 9` |
| Дата из момента | `значение::date` | `scheduled_departure::date = '2016-10-20'` |
| Поле из контактов пассажира | `contact_data->>'поле'` | `contact_data->>'email'`, `contact_data->>'phone'` |
| Значение из списка | `IN (…)` | `city IN ('Москва', 'Казань')` |
| Значение из диапазона, включая границы | `BETWEEN … AND …` | `range BETWEEN 3000 AND 6000` |
| Пропуск строк перед выводом | `LIMIT … OFFSET …` | `LIMIT 5 OFFSET 10` — строки с 11-й по 15-ю |
| Пустое значение | `IS NULL`, `IS NOT NULL`, `COALESCE(a, b)` | `COALESCE(NULL, 5)` → `5` |
| Строковые функции | `initcap`, `split_part` | `initcap('IVAN')` → `Ivan`, `split_part('IVAN PETROV', ' ', 1)` → `IVAN` |

### Задача 3. Знакомство с таблицами (0,5 балла)

1. Выведите 10 строк из таблицы билетов `bookings.tickets` — все столбцы.
2. Выведите все уникальные классы обслуживания из таблицы мест `bookings.seats`.

In [ ]:
%%sql
-- Задача 3.1. Напишите запрос

In [ ]:
%%sql
-- Задача 3.2. Напишите запрос

### Задача 4. Первое и самые дорогие бронирования (1 балл)

Работаем с таблицей бронирований `bookings.bookings`. Агрегатные функции (`min`, `max`) мы ещё не проходили —
решите задачу сортировкой.

1. Найдите момент, когда было оформлено самое первое бронирование. Выведите один столбец **book_date**, одну строку.
2. Выведите пять самых дорогих бронирований: **book_ref**, **book_date**, **total_amount** —
   от самого дорогого к менее дорогим.
3. Выведите бронирования, которые занимают **места с 6-го по 10-е** по стоимости. Столбцы и порядок те же.

In [ ]:
%%sql
-- Задача 4.1. Напишите запрос

In [ ]:
%%sql
-- Задача 4.2. Напишите запрос

In [ ]:
%%sql
-- Задача 4.3. Напишите запрос

### Задача 5. Пассажиры с годом рождения в почте (1 балл)

У части пассажиров адрес почты заканчивается годом рождения, например `bondarenko.ilya_1980@postgrespro.ru`.

Найдите пассажиров, у которых в адресе почты прямо перед знаком `@` стоит `1980`.
Выведите **ticket_no**, **passenger_name**, **email** — первые 10 строк по номеру билета.

In [ ]:
%%sql
-- Задача 5. Напишите запрос

### Задача 6. Дальность полёта самолётов (1 балл)

Работаем с таблицей `bookings.aircrafts`.

1. Выведите самолёты, которые летают очень далеко или очень близко: дальность **больше 7000 км
   или меньше 3000 км**. Столбцы: **aircraft_code**, **model**, **range**.
2. Выведите **все** самолёты с новым столбцом **range_group**: `ближний`, если дальность меньше 3000 км;
   `дальний`, если больше 7000 км; `средний` — в остальных случаях.
   Столбцы: **model**, **range**, **range_group**, сортировка по дальности.

In [ ]:
%%sql
-- Задача 6.1. Напишите запрос

In [ ]:
%%sql
-- Задача 6.2. Напишите запрос

### Задача 7. Рейсы из Казани за сентябрь 2016 (1 балл)

Какие рейсы из Казани (в базе у неё один аэропорт, код `KZN`) стояли в расписании на сентябрь 2016 года?

Отбирайте по времени вылета по расписанию — `scheduled_departure`: месяц и год достаньте из него
функцией для работы с частями даты.

Выведите **flight_id**, **flight_no**, **scheduled_departure**, **arrival_airport**,
отсортируйте по времени вылета.

In [ ]:
%%sql
-- Задача 7. Напишите запрос

### Задача 8. Текст из столбцов (1 балл)

1. Для выпадающего списка на сайте нужен текст вида `Город (КОД)`, например `Абакан (ABA)`.
   Выведите для всех аэропортов **airport_code**, **city** и новый столбец **airport_label**
   с таким текстом, отсортируйте по **airport_label**.
2. В таблице билетов имя и фамилия пассажира записаны одной строкой заглавными буквами: `VALERIY TIKHONOV`.
   Для билетов бронирования `06B046` выведите **ticket_no**, **passenger_name** и новый столбец
   **last_name** — только фамилию в виде `Tikhonov`.

In [ ]:
%%sql
-- Задача 8.1. Напишите запрос

In [ ]:
%%sql
-- Задача 8.2. Напишите запрос

### Задача 9. Рейсы с большим опозданием (1 балл)

Найдите рейсы, у которых вылет по расписанию (`scheduled_departure`) приходится на
**23 сентября 2016 года**, а прилёт состоялся **позже расписания больше чем на час**.

Выведите **flight_id**, **flight_no**, **scheduled_arrival**, **actual_arrival** и новый столбец
**delay** — на сколько рейс опоздал. Отсортируйте от самого большого опоздания к меньшим.

In [ ]:
%%sql
-- Задача 9. Напишите запрос

### Задача 10. Табло вылетов Москвы (1 балл)

Соберём табло вылетов из Москвы. В базе у Москвы три аэропорта: Шереметьево (`SVO`),
Домодедово (`DME`) и Внуково (`VKO`). На табло попадают рейсы, у которых вылет по расписанию (`scheduled_departure`)
приходится на промежуток **с 15:00 до 19:00 13 октября 2016 года**. Данные в базе обрываются
в 17:00 этого дня, поэтому часть рейсов ещё не вылетела.

1. Выведите рейсы, которые **ещё не вылетели**: у них не заполнено фактическое время вылета.
   Столбцы: **flight_no**, **departure_airport**, **scheduled_departure**, **status**,
   сортировка по времени вылета по расписанию.
2. Выведите **все** рейсы табло с новым столбцом **departure** — время вылета: фактическое,
   а если рейс ещё не вылетел — по расписанию.
   Столбцы: **flight_no**, **departure_airport**, **status**, **departure**, сортировка по **departure**.

In [ ]:
%%sql
-- Задача 10.1. Напишите запрос

In [ ]:
%%sql
-- Задача 10.2. Напишите запрос